# HW2: Phoneme Classification

Task: classify each speech frame into one of 41 phoneme classes.

- Training data: 3429 preprocessed audio feature sequences with labels.
- Testing data: 857 preprocessed audio feature sequences without labels.
- Training frames: 2116794.
- Testing frames: 527364.
- Metric: categorical accuracy.

## Download the Dataset

In [ ]:
# !pip install kagglehub==1.0.1

import inspect
import os
from pathlib import Path
from zipfile import ZipFile

DEFAULT_DOWNLOAD_DIR = Path("machine-learning/hung-yi-lee-machine-learning/HW2")
if not DEFAULT_DOWNLOAD_DIR.exists():
    DEFAULT_DOWNLOAD_DIR = Path.cwd()
DEFAULT_DOWNLOAD_DIR = DEFAULT_DOWNLOAD_DIR.resolve()
DATA_DOWNLOAD_DIR = DEFAULT_DOWNLOAD_DIR / "data"
DATA_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Optional: set this to a local dataset directory if the competition files were downloaded manually.
DATA_DOWNLOAD_ROOT = None
# DATA_DOWNLOAD_ROOT = "/path/to/ml2023spring-hw2"

# Optional: set this if the token is not under the current user's home directory.
KAGGLE_CREDENTIALS_DIR = None
# KAGGLE_CREDENTIALS_DIR = "/path/to/.kaggle"

KAGGLE_CONFIG_DIR = Path(
    KAGGLE_CREDENTIALS_DIR
    or os.environ.get("KAGGLE_CONFIG_DIR", Path.home() / ".kaggle")
).expanduser().resolve()
KAGGLE_ACCESS_TOKEN = KAGGLE_CONFIG_DIR / "access_token"
KAGGLE_JSON = KAGGLE_CONFIG_DIR / "kaggle.json"
os.environ["KAGGLE_CONFIG_DIR"] = str(KAGGLE_CONFIG_DIR)
os.environ["KAGGLEHUB_CACHE"] = str(DATA_DOWNLOAD_DIR)


def find_existing_data_dir(root: Path) -> Path | None:
    candidates = [root, root / "libriphone"]
    if root.exists():
        candidates.extend(root.glob("**/libriphone"))

    for candidate in candidates:
        if (candidate / "train_split.txt").exists() and (candidate / "feat" / "train").exists():
            return candidate.resolve()
    return None

def has_kaggle_credentials() -> bool:
    has_env_credentials = bool(os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
    has_access_token = KAGGLE_ACCESS_TOKEN.exists()
    has_json_credentials = KAGGLE_JSON.exists()
    if has_access_token:
        KAGGLE_ACCESS_TOKEN.chmod(0o600)
    if has_json_credentials:
        KAGGLE_JSON.chmod(0o600)
    return has_env_credentials or has_access_token or has_json_credentials


def download_competition(handle: str, output_dir: Path) -> str:
    import kagglehub

    signature = inspect.signature(kagglehub.competition_download)
    if "output_dir" in signature.parameters:
        return kagglehub.competition_download(handle, output_dir=str(output_dir))
    return kagglehub.competition_download(handle)


if DATA_DOWNLOAD_ROOT is None:
    existing_data_dir = find_existing_data_dir(DATA_DOWNLOAD_DIR)
    if existing_data_dir is not None:
        download_path = existing_data_dir
        path = str(download_path)
    else:
        if not has_kaggle_credentials():
            raise FileNotFoundError(
                f"Kaggle credentials were not found. Put the KaggleHub token at {KAGGLE_ACCESS_TOKEN}, "
                f"put legacy API credentials at {KAGGLE_JSON}, or set KAGGLE_USERNAME and KAGGLE_KEY "
                "before starting this notebook. Do not paste the token into this notebook."
            )

        try:
            downloaded_root = Path(download_competition("ml2023spring-hw2", DATA_DOWNLOAD_DIR)).expanduser().resolve()
        except Exception as error:
            error_name = error.__class__.__name__
            error_text = str(error)
            if error_name == "UnauthenticatedError":
                raise RuntimeError(
                    "KaggleHub still cannot authenticate. Make sure the current Jupyter kernel can see "
                    f"{KAGGLE_ACCESS_TOKEN} or {KAGGLE_JSON}, or restart the kernel after setting environment credentials. "
                    "Also join the competition and accept its rules on Kaggle before re-running this cell."
                ) from error
            if "403" in error_text and "rules" in error_text.lower():
                raise PermissionError(
                    "Kaggle authentication succeeded, but this account does not have permission to download "
                    "the competition data yet. Open https://www.kaggle.com/competitions/ml2023spring-hw2/rules, "
                    "join the competition, accept the rules, then re-run this cell."
                ) from error
            raise

        for zip_path in downloaded_root.glob("*.zip"):
            target_dir = downloaded_root / zip_path.stem
            if not target_dir.exists():
                with ZipFile(zip_path) as archive:
                    archive.extractall(target_dir)

        download_path = find_existing_data_dir(downloaded_root) or find_existing_data_dir(DATA_DOWNLOAD_DIR)
        if download_path is None:
            raise FileNotFoundError("Could not find the downloaded libriphone data directory.")
        path = str(download_path)
else:
    download_path = Path(DATA_DOWNLOAD_ROOT).expanduser().resolve()
    path = str(download_path)

print("Data directory:", download_path)
print("KaggleHub cache:", os.environ["KAGGLEHUB_CACHE"])


## Setup and Model Overview

Frame-level classification uses local context. For each frame $x_t\in\mathbb{R}^d$, concatenate neighboring frames:

$$
\tilde{x}_t=[x_{t-k},\dots,x_t,\dots,x_{t+k}]\in\mathbb{R}^{(2k+1)d}.
$$

The model estimates

$$
p(y_t=c\mid \tilde{x}_t)=\operatorname{softmax}(f_\theta(\tilde{x}_t))_c.
$$

Training minimizes cross entropy over labeled frames.

In [ ]:
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

try:
    import wandb
except ImportError:
    wandb = None


@dataclass
class Config:
    seed: int = 42
    validation_ratio: float = 0.1
    concat_nframes: int = 11
    batch_size: int = 1024
    hidden_dims: tuple[int, ...] = (512, 512, 256)
    dropout: float = 0.2
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    max_epochs: int = 30
    early_stopping_patience: int = 5
    num_classes: int = 41
    num_workers: int = 0
    gpu_id: int = 0
    model_filename: str = "phoneme_classifier.pt"
    submission_filename: str = "submission_hw2.csv"
    use_wandb: bool = False
    wandb_project: str = "hung-yi-lee-machine-learning"
    wandb_run_name: str = "hw2-phoneme-classification"


CONFIG = Config()


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def choose_device() -> torch.device:
    if torch.cuda.is_available():
        gpu_count = torch.cuda.device_count()
        gpu_id = CONFIG.gpu_id if 0 <= CONFIG.gpu_id < gpu_count else 0
        if gpu_id != CONFIG.gpu_id:
            print(f"Requested CUDA GPU {CONFIG.gpu_id}, but only {gpu_count} GPU(s) are available. Falling back to cuda:0.")
        if hasattr(torch, "set_float32_matmul_precision"):
            torch.set_float32_matmul_precision("high")
        torch.backends.cudnn.benchmark = True
        device = torch.device(f"cuda:{gpu_id}")
        torch.cuda.set_device(device)
        print(f"Using CUDA GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)}")
        return device

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("Using Apple Metal Performance Shaders (MPS).")
        return torch.device("mps")

    print("Using CPU. GPU acceleration is not available in this environment.")
    return torch.device("cpu")


def to_device(tensor: torch.Tensor) -> torch.Tensor:
    return tensor.to(device, non_blocking=(device.type == "cuda"))


def dataloader_kwargs(shuffle: bool = False) -> dict:
    return {
        "batch_size": CONFIG.batch_size,
        "shuffle": shuffle,
        "num_workers": CONFIG.num_workers,
        "pin_memory": device.type == "cuda",
    }


def init_wandb():
    if not CONFIG.use_wandb:
        return None
    if wandb is None:
        raise ImportError("Install wandb or set CONFIG.use_wandb=False.")
    return wandb.init(
        project=CONFIG.wandb_project,
        name=CONFIG.wandb_run_name,
        config=asdict(CONFIG),
    )


set_seed(CONFIG.seed)
device = choose_device()
wandb_run = init_wandb()

print(f"Using device: {device}")
print(asdict(CONFIG))

## Locate and Inspect the Dataset

In [ ]:
def find_data_dir() -> Path:
    roots: list[Path] = [Path.cwd(), Path.cwd().parent, Path(".").resolve()]
    if "path" in globals():
        roots.append(Path(path))

    candidates: list[Path] = []
    for root in roots:
        candidates.extend([root, root / "libriphone"])
        if root.exists():
            candidates.extend(root.glob("**/libriphone"))

    for candidate in candidates:
        if (candidate / "train_split.txt").exists() and (candidate / "feat" / "train").exists():
            return candidate.resolve()

    raise FileNotFoundError("Could not find the libriphone data directory. Run the Kaggle download cell first.")


def read_split(data_dir: Path, names: tuple[str, ...]) -> Optional[list[str]]:
    for name in names:
        split_path = data_dir / name
        if split_path.exists():
            return split_path.read_text().splitlines()
    return None


def read_label_map(data_dir: Path) -> dict[str, torch.Tensor]:
    label_map: dict[str, torch.Tensor] = {}
    for line in (data_dir / "train_labels.txt").read_text().splitlines():
        parts = line.split()
        if not parts:
            continue
        label_map[parts[0]] = torch.tensor([int(label) for label in parts[1:]], dtype=torch.long)
    return label_map


DATA_DIR = find_data_dir()
TRAIN_FEAT_DIR = DATA_DIR / "feat" / "train"
TEST_FEAT_DIR = DATA_DIR / "feat" / "test"

train_split = read_split(DATA_DIR, ("train_split.txt",))
valid_split = read_split(DATA_DIR, ("valid_split.txt", "val_split.txt"))
test_split = read_split(DATA_DIR, ("test_split.txt",))
label_map = read_label_map(DATA_DIR)

if train_split is None or test_split is None:
    raise FileNotFoundError("Expected train_split.txt and test_split.txt in the data directory.")

if valid_split is None:
    rng = np.random.default_rng(CONFIG.seed)
    shuffled = np.array(train_split)
    rng.shuffle(shuffled)
    valid_size = int(len(shuffled) * CONFIG.validation_ratio)
    valid_split = shuffled[:valid_size].tolist()
    train_split = shuffled[valid_size:].tolist()

print("Data directory:", DATA_DIR)
print("Train utterances:", len(train_split))
print("Valid utterances:", len(valid_split))
print("Test utterances:", len(test_split))
print("Label sequences:", len(label_map))

## Prepare Frame-Level Tensors

In [ ]:
def load_feature(feat_dir: Path, utterance_id: str) -> torch.Tensor:
    feat_path = feat_dir / f"{utterance_id}.pt"
    return torch.load(feat_path, map_location="cpu").float()


def concat_frames(feat: torch.Tensor, nframes: int) -> torch.Tensor:
    if nframes == 1:
        return feat
    if nframes % 2 == 0:
        raise ValueError("concat_nframes must be odd.")

    pad = nframes // 2
    left = feat[:1].repeat(pad, 1)
    right = feat[-1:].repeat(pad, 1)
    padded = torch.cat([left, feat, right], dim=0)
    frames = [padded[offset : offset + feat.shape[0]] for offset in range(nframes)]
    return torch.cat(frames, dim=1)


def preprocess_split(
    split: list[str],
    feat_dir: Path,
    labels: Optional[dict[str, torch.Tensor]] = None,
    concat_nframes: int = 1,
) -> tuple[torch.Tensor, Optional[torch.Tensor]]:
    features: list[torch.Tensor] = []
    targets: list[torch.Tensor] = []

    for utterance_id in tqdm(split, desc="Preprocessing"):
        feat = concat_frames(load_feature(feat_dir, utterance_id), concat_nframes)
        features.append(feat)

        if labels is not None:
            target = labels[utterance_id]
            if len(target) != len(feat):
                raise ValueError(f"Feature and label lengths differ for {utterance_id}.")
            targets.append(target)

    x = torch.cat(features, dim=0)
    y = torch.cat(targets, dim=0) if labels is not None else None
    return x, y


train_x, train_y = preprocess_split(train_split, TRAIN_FEAT_DIR, label_map, CONFIG.concat_nframes)
valid_x, valid_y = preprocess_split(valid_split, TRAIN_FEAT_DIR, label_map, CONFIG.concat_nframes)
test_x, _ = preprocess_split(test_split, TEST_FEAT_DIR, None, CONFIG.concat_nframes)

print("Train:", tuple(train_x.shape), tuple(train_y.shape))
print("Valid:", tuple(valid_x.shape), tuple(valid_y.shape))
print("Test:", tuple(test_x.shape))

In [ ]:
train_loader = DataLoader(
    TensorDataset(train_x, train_y),
    batch_size=CONFIG.batch_size,
    shuffle=True,
    pin_memory=torch.cuda.is_available(),
)
valid_loader = DataLoader(
    TensorDataset(valid_x, valid_y),
    batch_size=CONFIG.batch_size,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    TensorDataset(test_x),
    batch_size=CONFIG.batch_size,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
)

input_dim = train_x.shape[1]
print("Input dimension:", input_dim)

## Build and Train the Classifier

In [ ]:
class PhonemeClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: tuple[int, ...], num_classes: int, dropout: float) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        current_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(current_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            current_dim = hidden_dim
        layers.append(nn.Linear(current_dim, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


model = PhonemeClassifier(input_dim, CONFIG.hidden_dims, CONFIG.num_classes, CONFIG.dropout).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG.learning_rate, weight_decay=CONFIG.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

trainable_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")
model

In [ ]:
def run_epoch(model: nn.Module, loader: DataLoader, training: bool) -> tuple[float, float]:
    model.train(training)
    total_loss = 0.0
    correct = 0
    total = 0

    for x_batch, y_batch in tqdm(loader, leave=False):
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        with torch.set_grad_enabled(training):
            logits = model(x_batch)
            loss = criterion(logits, y_batch)

        if training:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * len(x_batch)
        correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total += len(x_batch)

    return total_loss / total, correct / total


history: list[dict[str, float]] = []
best_valid_accuracy = 0.0
epochs_without_improvement = 0
checkpoint_path = Path(CONFIG.model_filename)

for epoch in range(1, CONFIG.max_epochs + 1):
    train_loss, train_accuracy = run_epoch(model, train_loader, training=True)
    valid_loss, valid_accuracy = run_epoch(model, valid_loader, training=False)
    scheduler.step(valid_accuracy)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "valid_loss": valid_loss,
        "valid_accuracy": valid_accuracy,
    })

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} train_acc={train_accuracy:.4f} | "
        f"valid_loss={valid_loss:.4f} valid_acc={valid_accuracy:.4f}"
    )

    if valid_accuracy > best_valid_accuracy:
        best_valid_accuracy = valid_accuracy
        epochs_without_improvement = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "config": asdict(CONFIG),
            "input_dim": input_dim,
            "valid_accuracy": valid_accuracy,
        }, checkpoint_path)
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= CONFIG.early_stopping_patience:
        print(f"Early stopping at epoch {epoch}.")
        break

checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Best validation accuracy: {checkpoint['valid_accuracy']:.4f}")

In [ ]:
history_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["valid_loss"], label="valid")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["train_accuracy"], label="train")
axes[1].plot(history_df["epoch"], history_df["valid_accuracy"], label="valid")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.show()

## Predict the Test Set and Export the Submission

In [ ]:
@torch.no_grad()
def predict(model: nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval()
    predictions: list[torch.Tensor] = []

    for (x_batch,) in tqdm(loader):
        logits = model(x_batch.to(device))
        predictions.append(logits.argmax(dim=1).cpu())

    return torch.cat(predictions).numpy()


test_predictions = predict(model, test_loader)
submission = pd.DataFrame({
    "Id": np.arange(len(test_predictions)),
    "Class": test_predictions,
})

submission_path = Path(CONFIG.submission_filename)
submission.to_csv(submission_path, index=False)

print(f"Saved submission to: {submission_path.resolve()}")
submission.head()